In [ ]:
# eval.ipynb
# Deferred evaluation: run AFTER run.ipynb finishes training, on the SAME pod.
# Training only emitted slim probe checkpoints to a queue (no probe ran during
# training, to avoid GPU contention). This notebook drains that queue (Prob),
# runs the final pooling-vs-VICReg eval, then archives everything.
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
LOG = "/workspace/stable_query_latent_logs/pipeline.log"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
QUEUE_DIR = OUT_DIR + "/probe_queue"
LOCAL_DATA = "/root/data"   # local NVMe: resident .dat + meta-H5 for the drain
os.makedirs(LOCAL_DATA, exist_ok=True)
print('repo :', REPO)
print('out  :', OUT_DIR)
print('log  :', LOG)
print('local:', LOCAL_DATA)


In [ ]:
# FORCE-sync to origin/main before evaluating (pod repo is a mirror of GitHub;
# local edits in tracked paths are discarded, untracked files untouched).
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD

In [ ]:
# Start GPU + CPU + RAM + disk I/O monitor (same as run.ipynb).
import subprocess, threading, time, psutil
from pathlib import Path

stop = False

def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == "max":
            return None
        return int(text)
    except Exception:
        return None

def get_memory_status():
    limit = _read_int("/sys/fs/cgroup/memory.max")
    used = _read_int("/sys/fs/cgroup/memory.current")
    if limit is None or used is None:
        limit = _read_int("/sys/fs/cgroup/memory/memory.limit_in_bytes")
        used = _read_int("/sys/fs/cgroup/memory/memory.usage_in_bytes")
    if limit and used and limit < 10**18:
        return used / limit * 100, used / 1024**3, limit / 1024**3, "cgroup"
    vm = psutil.virtual_memory()
    return vm.percent, vm.used / 1024**3, vm.total / 1024**3, "host"

def get_gpu_status():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()
        parts = [p.strip() for p in out.splitlines()[0].split(",")]
        return f"{float(parts[0]):.0f}%, {float(parts[1])/1024:.1f}/{float(parts[2])/1024:.1f} GiB"
    except Exception as e:
        return f"n/a ({e})"

def monitor(interval=5):
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)
    while not stop:
        gpu = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)
        ram_pct, ram_used, ram_total, ram_source = get_memory_status()
        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)
        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t
        print(
            f"[gpu] {gpu} | [cpu] {cpu:.0f}% | "
            f"[ram:{ram_source}] {ram_pct:.0f}% ({ram_used:.1f}/{ram_total:.1f} GiB) | "
            f"[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s",
            flush=True,
        )
        time.sleep(interval)

threading.Thread(target=monitor, daemon=True).start()


In [ ]:
# Clear GPU memory.
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Requeue previously FAILED probe jobs (no-op when none failed).
# probe_worker renames a marker to *.json.failed + writes *.error.txt on
# failure; nothing is deleted, so after the cause is fixed (e.g. the pre-fix
# vector-source resolution) renaming them back to *.json re-enters the queue
# and the next Prob run picks them up.
from pathlib import Path

QUEUE = Path(REPO) / QUEUE_DIR

failed = sorted(QUEUE.glob('*.json.failed'))
for f in failed:
    f.rename(f.with_name(f.name[: -len('.failed')]))
for e in QUEUE.glob('*.error.txt'):
    e.unlink()

print(f'requeued : {len(failed)} failed probe job(s); error notes cleared')
print(f'pending  : {len(list(QUEUE.glob("*.json")))} marker(s) ready for the Prob cell')
print(f'done     : {len(list(QUEUE.glob("*.json.done")))} already processed')

In [ ]:
# Stage the resident vectors .dat locally for the drain (idempotent; disk-gated).
# extract_features re-reads sentence vectors for EVERY marker -- over the network
# FS that is the drain's bottleneck. One local staging (~150GiB read, reused when
# already valid, e.g. on a pod that trained) lets every probe gather from NVMe
# exactly like training. Skipped when the local disk can't hold the .dat; the
# worker then falls back to the shared H5 (slow but correct).
import shutil, sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from VICReg_review import train_vicreg_review_h5 as _T

WORKSPACE_H5 = f'{REPO}/game_review_data/embedding_h5.h5'
GIB = 1024 ** 3
_rows, _dim, _dt = _T.resident_vectors_meta(WORKSPACE_H5)
_vbytes = _rows * _dim * (2 if _dt == 'float16' else 4)
_have = _T.resident_descriptor(
    _T.default_vectors_dat_path(WORKSPACE_H5, LOCAL_DATA), WORKSPACE_H5) is not None
_free = shutil.disk_usage(LOCAL_DATA).free
if _have or _free >= _vbytes * 1.05:
    _dat = _T.stage_resident_vectors(WORKSPACE_H5, work_dir=LOCAL_DATA)   # reuses a valid .dat
    _meta = _T.write_offsets_h5(WORKSPACE_H5, _T.default_offsets_h5_path(WORKSPACE_H5, LOCAL_DATA))
    print(f'drain vectors : {_dat[0]}  shape={_dat[1]}  (local NVMe)')
    print(f'drain meta    : {_meta}')
else:
    print(f'resident SKIP: need {_vbytes * 1.05 / GIB:.0f}GiB free on {LOCAL_DATA}, '
          f'have {_free / GIB:.0f}GiB -> drain will read the shared H5 (slower)')

In [ ]:
# Prob: drain the probe queue in ONE pass (training is done, so the backlog is
# complete and static). This fills every <combo>/dual_probe_history.tsv with the
# convergence curve. GPU is fully free now, so there is no training contention.
# Vector source ladder (probe_worker.resolve_vector_source):
#   1. the training pod's own .dat (markers drained where they were made);
#   2. the .dat staged locally by the cell above (--local-data-dir) -> NVMe
#      gather on ANY pod, no per-marker network reads;
#   3. a marker's input_h5 when it holds real vectors (streaming-mode combos);
#   4. --fallback-h5 (shared source H5) -- slow but correct anywhere;
#   never the resident meta-H5's shape-only zeros.
!python -u VICReg_review/probe_worker.py \
  --queue-dir {QUEUE_DIR} \
  --device cuda \
  --run-once \
  --fallback-h5 game_review_data/embedding_h5.h5 \
  --local-data-dir {LOCAL_DATA} \
  --logout-address {LOG}


In [ ]:
# Clear GPU memory.
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Final eval: pooling-vs-VICReg comparison on the best checkpoint.
# --skip-train skips all training; --eval-mode final_best runs the comparison.
# --calib-mode off skips the (training-only) OOM calibration.
#
# Grid flags are DERIVED from sweep.yaml's FULL grid (grid.exclude cleared),
# never hand-written: the eval side enumerates by CLI flags, not the yaml, so
# masked cells (e.g. the excluded 512/1024 x view80 corners) are still swept
# for artifacts -- their DONE checkpoints count as experiment data. Missing or
# unfinished combos are skipped by the manifest status=done checks, never an
# error. Editing sweep.yaml automatically keeps this cell in sync.
#
# Training claims larger num/train-game-count tiers first. Eval mirrors that by
# passing train-game-counts in biggest-first order, so it reads the biggest batch
# before falling through to smaller completed batches. 0 means all games and is
# ordered before finite counts.
import copy, sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from VICReg_review.sweep.config import SweepConfig


def _count_value(n):
    if isinstance(n, str) and n.strip().lower() == 'all':
        return 0
    return int(n)


def _biggest_first_counts(counts):
    values = [_count_value(n) for n in counts]
    return sorted(values, key=lambda n: (n <= 0, n), reverse=True)


_cfg = SweepConfig.load(f'{REPO}/VICReg_review/sweep/sweep.yaml')
_full = copy.deepcopy(_cfg)
_full.grid.exclude = []
_g = _full.grid
GAME_COUNTS = ' '.join(str(n) for n in _biggest_first_counts(_g.train_game_counts))
VIEWS = ' '.join(f'{float(v):g}' for v in _g.sample_fractions)
DIMS = ' '.join(str(int(d)) for d in _g.output_dims)
SCALES = ' '.join(f'{float(s):g}' for s in _g.latent_scales)
EXPANDER_HIDDEN = ' '.join(str(int(h)) for h in _cfg.model.expander_hidden)
ANCHORS = ','.join(str(a) for a in _cfg.data_seed.anchors)
print('grid flags from sweep.yaml (FULL grid, exclude cleared; n biggest-first):')
print(f'  n=[{GAME_COUNTS}]  views=[{VIEWS}]  dims=[{DIMS}]  scales=[{SCALES}]')

!python -u VICReg_review/sweep_cloud.py \
  --h5 game_review_data/embedding_h5.h5 \
  --out-dir {OUT_DIR} \
  --train-game-counts {GAME_COUNTS} \
  --sample-fractions {VIEWS} \
  --output-dims {DIMS} \
  --latent-scales {SCALES} \
  --base-num-latents {_g.base_num_latents} \
  --expander-dim {_cfg.model.expander_dim} \
  --expander-hidden {EXPANDER_HIDDEN} \
  --epochs {_cfg.train.epochs} \
  --batch-size {_cfg.train.batch_size} \
  --skip-train \
  --eval-mode final_best \
  --calib-mode off \
  --probe-queue-dir {QUEUE_DIR} \
  --train-game-anchor-appids "{ANCHORS}" \
  --logout-address {LOG}


In [ ]:
# Collect outputs (archive heads, H5s, manifests + paper handoff).
stop = True

from datetime import datetime
from pathlib import Path
import copy
import json
import subprocess
import shutil
import sys

repo = Path(REPO)
out = Path('/workspace') / 'stable_query_latent_artifacts' / datetime.now().strftime('%Y%m%d_%H%M%S')
out.mkdir(parents=True, exist_ok=True)

def run_capture(cmd):
    return subprocess.run(cmd, cwd=repo, capture_output=True, text=True).stdout.strip()

def copy_item(rel: str):
    src = repo / rel
    dst = out / rel
    if not src.exists():
        print(f'skip missing: {src}')
        return None
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        kind = 'dir'
    else:
        shutil.copy2(src, dst)
        kind = 'file'
    print(f'copied: {rel}')
    return {'item': rel, 'kind': kind}

# FULL_ARCHIVE=True also copies the immutable input H5s (~200GB) into the
# archive -- use for the FINAL collection. False = interim run: heads +
# manifests + status only; the H5s never change, so any one full archive
# already preserves them.
FULL_ARCHIVE = True

items = [
    'VICReg_review/heads',
    'game_review_data/embedding_h5.h5.incloud_manifest.json',
    'game_review_data/build_new_gamedata/text_h5.h5.manifest.json',
]
if FULL_ARCHIVE:
    items += [
        'game_review_data/embedding_h5.h5',
        'game_review_data/build_new_gamedata/text_h5.h5',
    ]
manifest = [x for x in (copy_item(rel) for rel in items) if x is not None]

# Combo status inventory over the FULL grid (grid.exclude cleared): every combo
# -- live or masked -- classified done / partial / not_started. Eval data
# selection filters on state=='done'; masked done combos ARE experiment data.
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from VICReg_review.sweep.config import SweepConfig

_cfg = SweepConfig.load(str(repo / 'VICReg_review/sweep/sweep.yaml'))
_full = copy.deepcopy(_cfg)
_full.grid.exclude = []
live_ids = {c.combo_id for c in _cfg.iter_combos()}
sweep_src = repo / OUT_DIR
combo_status = []
for c in _full.iter_combos():
    d = sweep_src / c.combo_id
    try:
        man = json.loads((d / 'vicreg_review_h5_manifest.json').read_text(encoding='utf-8'))
    except Exception:
        man = {}
    done = (man.get('status') == 'done') or (d / 'done.json').exists()
    state = ('done' if done else
             'partial' if (d / 'vicreg_review_h5_latest.pt').exists() else
             'not_started')
    combo_status.append({
        'combo_id': c.combo_id,
        'masked': c.combo_id not in live_ids,
        'state': state,
        'epoch': man.get('epoch'),
        'output_dim': c.output_dim, 'num_latents': c.num_latents,
        'train_games': c.train_games, 'view': c.view, 'arm': c.arm,
    })
(out / 'combo_status.json').write_text(
    json.dumps(combo_status, ensure_ascii=False, indent=1), encoding='utf-8')
_tally = {}
for row in combo_status:
    key = ('masked' if row['masked'] else 'live', row['state'])
    _tally[key] = _tally.get(key, 0) + 1
print('combo status inventory (full grid):')
for (scope, state), n in sorted(_tally.items()):
    print(f'  {scope:6} {state:11} : {n}')
usable = sum(1 for r in combo_status if r['state'] == 'done')
print(f'  -> collectable as experiment data (state=done, live+masked): {usable}')

log_src = Path(LOG)
if log_src.exists():
    log_dst = out / 'pipeline.log'
    shutil.copy2(log_src, log_dst)
    manifest.append({'item': str(log_src), 'kind': 'file', 'copied_as': 'pipeline.log'})
    print(f'copied log: {log_src}')
else:
    print(f'skip missing log: {log_src}')

git_info = {
    'commit': run_capture(['git', 'rev-parse', 'HEAD']),
    'branch': run_capture(['git', 'branch', '--show-current']),
    'status_short': run_capture(['git', 'status', '--short']),
    'remote': run_capture(['git', 'remote', '-v']),
}
(out / 'git_info.json').write_text(json.dumps(git_info, ensure_ascii=False, indent=2), encoding='utf-8')

sweep_dir = out / 'VICReg_review' / 'heads' / 'cloud_full_sweep_a100'
paper_files = [
    'DATA_VIEW_SWEEP_REPORT.md',
    'data_view_sweep_summary.csv',
    'data_view_sweep_summary.json',
    'sweep_manifest.json',
    'calib.json',
    # final_best eval-mode writes the real pooling-vs-VICReg comparison here,
    # not into per-combo eval_report.json files, so point the handoff at it.
    'final_best_eval/final_best_eval.json',
    'final_best_eval/eval_report.json',
    'raw_test_data/training_manifests.csv',
    'raw_test_data/probe_summary.csv',
    'raw_test_data/recommendation_probe.csv',
    'raw_test_data/identity_summary.csv',
    'raw_test_data/identity_retrieval_details.csv',
    'raw_test_data/identity_pair_cosine_details.csv',
    'raw_test_data/tag_freq_floor_details.csv',
    'raw_test_data/tag_top_bottom_details.csv',
    'raw_test_data/tag_fold_details.csv',
]
available_paper_files = [rel for rel in paper_files if (sweep_dir / rel).exists()]
missing_paper_files = [rel for rel in paper_files if not (sweep_dir / rel).exists()]

# Per-combo in-training convergence histories (one row per probe epoch).
probe_history_files = sorted(
    str(p.relative_to(sweep_dir)).replace('\\', '/')
    for p in sweep_dir.glob('*/dual_probe_history.tsv')
)

handoff = [
    '# Paper Handoff',
    '',
    f'- Archive: `{out}`',
    f'- Git commit: `{git_info["commit"]}`',
    f'- Sweep dir: `{sweep_dir}`',
    f'- Pipeline log: `pipeline.log`' if (out / 'pipeline.log').exists() else '- Pipeline log: missing',
    '',
    '## Start Here',
    '',
    '1. Read `VICReg_review/heads/cloud_full_sweep_a100/DATA_VIEW_SWEEP_REPORT.md` for the high-level result.',
    '2. Read `final_best_eval/final_best_eval.json` for the pooling-vs-VICReg comparison on the best full-train model (the actual probe/identity numbers).',
    '3. Use `data_view_sweep_summary.csv` + `raw_test_data/training_manifests.csv` for the data-size x architecture grid (training loss/convergence per combo).',
    '4. Use `<combo>/dual_probe_history.tsv` for per-combo in-training convergence curves (sentiment/recommendation/tag probes every few epochs).',
    '5. Check `sweep_manifest.json` before interpreting incomplete runs; `calib.json` records the OOM-budget calibration used.',
    '6. `combo_status.json` inventories the FULL grid (masked cells included): filter `state=="done"` to collect experiment data -- masked done combos are valid data; partial/not_started are not.',
    '',
    '## Available Paper Files',
    '',
]
handoff.extend(f'- `{rel}`' for rel in available_paper_files)
if missing_paper_files:
    handoff.extend(['', '## Missing Or Not Yet Generated', ''])
    handoff.extend(f'- `{rel}`' for rel in missing_paper_files)
handoff.extend([
    '',
    '## Convergence Probe Histories',
    '',
    f'- {len(probe_history_files)} per-combo files at `<combo>/dual_probe_history.tsv` '
    '(full list in `collection_manifest.json` -> `probe_history_files`).',
])
(out / 'paper_handoff.md').write_text('\n'.join(handoff) + '\n', encoding='utf-8')

(out / 'collection_manifest.json').write_text(
    json.dumps({'repo': str(repo), 'archive': str(out), 'full_archive': FULL_ARCHIVE, 'git': git_info, 'items': manifest, 'paper_files': available_paper_files, 'missing_paper_files': missing_paper_files, 'probe_history_files': probe_history_files, 'combo_status_summary': {f'{scope}/{state}': n for (scope, state), n in sorted(_tally.items())}}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'archive ready: {out}')
print(f'probe history files: {len(probe_history_files)}')
